# MT Hexapod current noise at faults

This notebook will search timestamps for M2Hex faults
and analyze the current noise when it faults.\
We will only consider Faults with error codes equal to 1\
For a robust analysis, this notebook anaylzes since the start of observations in April

Noise being the standard deviation of the current in the motors, given by the EfdClient

M2Hex has several faults associated with motor oscillations. We want to characterize these oscillations/vibrations.\
I suggest the creation of a function that applies FFT to a motor current and extracts the most dominant frequency components.\
I will leave the technical aspects of the implementation open for now since there might be some pre-processing we need to apply to the data.

## General Data

First, lest stablish some general variables.

We'll start querying the last month, then we can change to what we see fit.\
Another important variable is the time window where the fault(s) happens.

# Pending
Warmups: does temp affects? does it really work? \
InPosition flag not working \
Correspond each error with its error type, InPosition flag, Temperature and Altitude\
\
Waterfall plot for frecuencies median\
same frecuencies in same day?\
See how this behaves on the big vibration on TEA (when was this?)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sb

from astropy.time import Time, TimeDelta
from scipy.fft import fft, ifft
from scipy.signal import detrend, find_peaks
from scipy.optimize import curve_fit

from lsst.summit.utils.efdUtils import getEfdData, getDayObsEndTime, getDayObsStartTime, makeEfdClient

import logging

In [ ]:
logger = logging.getLogger("m2hex_feq_analysis")

In [ ]:
logger.setLevel("DEBUG")

In [ ]:
logger.setLevel("DEBUG")

x = 12345
logger.debug(f"This is a debug message {x}")
logger.info("This is a info message")
logger.warning("This is a warning message")
logger.error("This is a error message")
logger.critical("This is a critical message")

In [ ]:
logger.info("This won't show")

In [ ]:
# Strut pairs     1    6    2    4    3    5
# motorCurrent    0    5    1    3    2    4

day_start = 20250730
day_end = 20250831


N_STRUTS = 6

error_code = 1 # Will only consider errorCode 1 as Faults
sal_index = 2 # M2Hexapods (maybe make it to search for CamHex faults aswell?)

# Upper and lower time difference from the fault
delta_start = 15
delta_end = 0

# Verbose and Save figures to see what the notebook is doing (don't understand logger yet)
verbose = True
save_figures = True
dpi = 350

# Pandas configuration
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Create the client for InfluxQL
efd_client = makeEfdClient()

## Query the timestamps

We need to know when do these faults happened

In [ ]:
start_time = getDayObsStartTime(day_start)
end_time = getDayObsEndTime(day_end)

# Query Hexapod faults
df_timestamps = getEfdData(
        client=efd_client,
        topic='lsst.sal.MTHexapod.logevent_errorCode',
        columns=["MTHexapodID", "errorCode", "errorReport", "salIndex", "traceback"],
        begin=start_time,
        end=end_time,
    )

df_timestamps = df_timestamps[df_timestamps.salIndex == sal_index]

# Know which Timestamps caused errorCode = 1 Faults
timestamps_error_code = df_timestamps[df_timestamps.errorCode == error_code].index

if verbose == True: print(timestamps_error_code)

#df_timestamps

In [ ]:
from lsst.ts.xml.enums.MTHexapod import ErrorCode as Codes
# Create a dataframe of all states
df_states = pd.DataFrame(
    [(state.value, state.name) for state in Codes],
    columns=["Number", "Name"]
)

# Display it
df_states

## Analysis of Current Noise

Now that we have the timestamps, we can query these timestamps and analyze the noise of the current in the proximity of these faults.

In [ ]:
def current_query(motor, delta_start, delta_end, timestamp):
    '''
    Query the noise current for a certain motor in a timelapse given by the deltas and timestamp
    Returns the format querys and the timestamp with delta_start substracted

    Parameters
    ----------
    motor: integer
        Strut ID for the motorCurrent to query
    delta_start: integer
        Time in seconds before the timestamp
    delta_end: integer
        Time in seconds after the timestamp
    timestamp: string (YYYY-MM-DD hh:mm:ss.ms)
        Time when error occurred
    '''

    timestamp = pd.to_datetime(timestamp)
    new_stamp = timestamp.value  
    step = int(0.01 * 1e10)
    rounded_stamp = (new_stamp // step) * step
    timestamp = Time(pd.to_datetime(rounded_stamp), scale="utc")
    
    delta1 = TimeDelta(delta_start, format='sec')
    delta2 = TimeDelta(delta_end, format='sec')
    
    start_time = (timestamp - delta1).to_value('isot', subfmt='date_hms') + 'Z'
    end_time = (timestamp + delta2).to_value('isot', subfmt='date_hms') + 'Z'
    
    start_plot = (timestamp - TimeDelta('60', format='sec')).to_value('isot', subfmt='date_hms') + 'Z'
    end_plot = (timestamp + TimeDelta('5', format='sec')).to_value('isot', subfmt='date_hms') + 'Z'
    
    error_query = f'''
            SELECT "motorCurrent{motor}"
            AS "Motor {motor} current"
            FROM "efd"."autogen"."lsst.sal.MTHexapod.electrical"
            WHERE time > '{start_time}'
            AND time < '{end_time}'
            '''
    
    context_query = f'''
            SELECT "motorCurrent{motor}"
            AS "Motor {motor} current"
            FROM "efd"."autogen"."lsst.sal.MTHexapod.electrical"
            WHERE time > '{start_plot}'
            AND time < '{end_plot}'
            '''
    
    return(error_query, context_query, start_time, end_time)


def fault_current_noise_query(motor, delta_start, delta_end, timestamp):
    '''
    Query the noise current for a certain motor in a timelapse given by the deltas and timestamp
    Returns the format querys and the timestamp with delta_start substracted

    Parameters
    ----------
    motor: integer
        Strut ID for the motorCurrent to query
    delta_start: integer
        Time in seconds before the timestamp
    delta_end: integer
        Time in seconds after the timestamp
    timestamp: string (YYYY-MM-DD hh:mm:ss.ms)
        Time when error occurred
    '''

    timestamp = pd.to_datetime(timestamp)
    new_stamp = timestamp.value  
    step = int(0.01 * 1e10)
    rounded_stamp = (new_stamp // step) * step
    timestamp = Time(pd.to_datetime(rounded_stamp), scale="utc")
    
    delta1 = TimeDelta(delta_start, format='sec')
    delta2 = TimeDelta(delta_end, format='sec')
    
    start_time = (timestamp - delta1).to_value('isot', subfmt='date_hms') + 'Z'
    end_time = (timestamp + delta2).to_value('isot', subfmt='date_hms') + 'Z'
    
    start_plot = (timestamp - TimeDelta('60', format='sec')).to_value('isot', subfmt='date_hms') + 'Z'
    end_plot = (timestamp + TimeDelta('5', format='sec')).to_value('isot', subfmt='date_hms') + 'Z'
    
    error_query = f'''
            SELECT stddev("motorCurrent{motor}")
            AS "STDDEV Motor current {motor} Error"
            FROM "efd"."autogen"."lsst.sal.MTHexapod.electrical"
            WHERE time > '{start_time}'
            AND time < '{end_time}'
            GROUP BY time(100ms) FILL(null)
            '''
    
    context_query = f'''
            SELECT stddev("motorCurrent{motor}")
            AS "STDDEV Motor current {motor}"
            FROM "efd"."autogen"."lsst.sal.MTHexapod.electrical"
            WHERE time > '{start_plot}'
            AND time < '{end_plot}'
            GROUP BY time(100ms) FILL(null)
            '''
    
    return(error_query, context_query, start_time, end_time)


def alt_az_query(delta_start, delta_end, timestamp):
    '''
    Query the altitude and azimuth for a given timestamp window
    Returns elevation and azimuths for the error window and context window
    
    Parameters
    ----------
    delta_start: integer
        Time in seconds before the timestamp
    delta_end: integer
        Time in seconds after the timestamp
    timestamp: string (YYYY-MM-DD hh:mm:ss.ms)
        Time when error occurred
    '''
    
    timestamp = pd.to_datetime(timestamp)
    new_stamp = timestamp.value  
    step = int(0.01 * 1e10)
    rounded_stamp = -(-new_stamp // step) * step
    timestamp = Time(pd.to_datetime(rounded_stamp), scale="utc")
    
    delta1 = TimeDelta(delta_start, format='sec')
    delta2 = TimeDelta(delta_end, format='sec')
    
    start_time = (timestamp - delta1).to_value('isot', subfmt='date_hms') + 'Z'
    end_time = (timestamp + delta2).to_value('isot', subfmt='date_hms') + 'Z'
    
    start_plot = (timestamp - TimeDelta('60', format='sec')).to_value('isot', subfmt='date_hms') + 'Z'
    end_plot = (timestamp + TimeDelta('5', format='sec')).to_value('isot', subfmt='date_hms') + 'Z'

    context_alt_query = f'''
            SELECT "actualPosition"
            AS "Elevation" 
            FROM "efd"."autogen"."lsst.sal.MTMount.elevation" 
            WHERE time > '{start_plot}' 
            AND time < '{end_plot}'
            '''
    context_az_query = f'''
            SELECT "actualPosition"
            AS "Azimuth" 
            FROM "efd"."autogen"."lsst.sal.MTMount.azimuth" 
            WHERE time > '{start_plot}' 
            AND time < '{end_plot}'
            '''
    error_alt_query = f'''
            SELECT "actualPosition"
            AS "Elevation" 
            FROM "efd"."autogen"."lsst.sal.MTMount.elevation" 
            WHERE time > '{start_time}' 
            AND time < '{end_time}'
            '''
    error_az_query = f'''
            SELECT "actualPosition"
            AS "Azimuth" 
            FROM "efd"."autogen"."lsst.sal.MTMount.azimuth" 
            WHERE time > '{start_time}' 
            AND time < '{end_time}'
            '''

    return(context_alt_query, context_az_query, error_alt_query, error_az_query)


def fft_positive_values(motor, df_currentmotor, dt):
    '''
    Computes and returns in dataframes the Positive values
    of a shifted FFT, of its frequencies and peaks indexes
    and of the detrended signal.
        
    Parameters
    ----------
    '''
    
    signal = df_currentmotor.values.flatten() 
    signal = detrend(signal)
    time = df_currentmotor.index
    
    fft_vals[motor] = np.fft.fftshift(np.fft.fft(signal))
    real_fft_vals[motor] = np.abs(np.real(fft_vals[motor]))
    imag_fft_vals[motor] = np.imag(fft_vals[motor])
    freqs[motor] = np.fft.fftshift(np.fft.fftfreq(len(signal), dt))
    mask = freqs[motor] > 0
    freqs[motor] = freqs[motor][mask]
    real_fft_vals[motor] = real_fft_vals[motor][mask]
    imag_fft_vals[motor] = imag_fft_vals[motor][mask]

    threshold = 0.5*np.max(np.abs(real_fft_vals[motor]))
    peaks_index[motor], properties[motor] = find_peaks(real_fft_vals[motor], height=threshold) # Modify height for distance

    df_signal[motor] = pd.DataFrame({
        'Time': time,
        f'Detrend signal': signal
    })
    df_signal[motor]["Time"] = pd.to_datetime(df_signal[motor]["Time"])
    df_signal[motor].set_index("Time", inplace=True)
    
    return(real_fft_vals[motor], imag_fft_vals[motor], freqs[motor], peaks_index[motor], df_signal[motor])

In [ ]:
median_val = {}
df_currentmotor = {}

for motor in range(N_STRUTS):
    median_val[motor] = []

timestamp_list = []
dt_list = []

# Cycle through timestamps and motors, then append each to their respective list
for timestamp in timestamps_error_code:
    timestamp_list.append(timestamp)
    
    for motor in range(N_STRUTS):
        
        median_query, context_query, start_time, end_time = current_query(motor, delta_start, delta_end, timestamp)
        df_currentmotor[motor] = await efd_client.influx_client.query(median_query)
        noisemedian = df_currentmotor[motor][f'Motor {motor} current'].median()
        median_val[motor].append(noisemedian)
        dt = df_currentmotor[motor].index.diff().median().total_seconds()
        dt_list.append(dt)

In [ ]:
plt.close()
median_df = {}

for motor in range(N_STRUTS):
    df = pd.DataFrame({
        'Time': timestamp_list,
        f'Median Noise {motor}': median_val[motor]
    })
    
    df['Time'] = pd.to_datetime(df['Time'])
    df.set_index('Time', inplace=True)
    median_df[motor] = df

ax = median_df[5].plot(figsize=(15,5), alpha=0.5, color='black')
median_df[4].plot(ax=ax, alpha=0.5, color='blue' )
median_df[3].plot(ax=ax, alpha=0.5, color='violet')
#median_df[2].plot(ax=ax, alpha=0.5, color='purple')
#median_df[1].plot(ax=ax, alpha=0.5, color='red')
#median_df[0].plot(ax=ax, alpha=0.5, color='orange')

plt.ylabel('Median of Noise current [A]')
plt.xlabel('Error timestamps')

plt.grid(True)
plt.show()
if save_figures:
    plt.savefig('Median_stddev_struts.png', dpi=dpi)

In [ ]:
plt.close()
plt.figure(figsize=(10,8))
signal_threshold = 0.5
bins=round((len(median_df[motor].index))/3)

for motor in range(N_STRUTS):
    plot_df = median_df[motor][median_df[motor][f'Median Noise {motor}'] > signal_threshold].index
    plt.hist(plot_df, bins=bins, alpha=0.5, label=f'Motor {motor}')

plt.xlabel(f'Time (Bins={bins})')
plt.ylabel('Frequency')
plt.xticks(rotation=20)
plt.title(f'Times when median noise > {signal_threshold}')
plt.legend()
plt.grid(True)
plt.show()
if save_figures:
    plt.savefig('Median_Histograms.png', dpi=dpi)

With these plots we can visually see that the possible main problem is the Strut 6, as the Current Noise is usually higher prior to a fault.
Also, with the time difference between samplings we can see a trend, and choose wether we use the median or the mean.

In [ ]:
bins=int(len(dt_list)*0.1)
dt_mean = np.mean(dt_list)
dt_median = np.median(dt_list)
dt_sigma = np.std(dt_list)

print(f' Mean: {dt_mean} \n Median: {dt_median} \n Sigma: {dt_sigma}')

plt.figure(figsize=(8,5))
plt.hist(dt_list, bins=bins, color='C1')
plt.xlabel('Sampling frequencies')
plt.ylabel('Ocassions')
plt.title('Histogram of sampling frequencies')
plt.grid(True)
plt.show()

## Oscillations of the current

To understand how the current behaves we have to see the components of the frecuency.

We're going back to one single timestamp.

## Not yet commented

In [ ]:
timestamp = '2025-06-06 23:35:50'

# First we stablish some dictionaries, then we create the queries for the timestamp(s)
df_contextquery = {}
df_currentmotor = {}
peaks_index = {}
properties = {}
df_signal = {}
fft_vals = {}
real_fft_vals = {}
imag_fft_vals = {}
freqs = {}
power = {}

context_alt, context_az, error_alt, error_az = alt_az_query(delta_start, delta_end, timestamp)

df_context_alt = await efd_client.influx_client.query(context_alt)
df_error_alt = await efd_client.influx_client.query(error_alt)
df_context_az = await efd_client.influx_client.query(context_az)
df_error_az = await efd_client.influx_client.query(error_az)

for motor in range(4,6):
    
    fft_query, context_query, start_time, end_time = current_query(motor, delta_start, delta_end, timestamp)
    
    df_contextquery[motor] = await efd_client.influx_client.query(context_query)
    df_currentmotor[motor] = await efd_client.influx_client.query(fft_query)

    dt = dt_median
    
    (real_fft_vals[motor], imag_fft_vals[motor],
     freqs[motor], peaks_index[motor], df_signal[motor]) = fft_positive_values(motor, df_currentmotor[motor], dt)
    
if verbose: # Plotting
    
    print(f'Time: {timestamp}')
        
    ax = df_contextquery[4].plot(figsize=(15,10), color='C0')
    df_currentmotor[4].plot(ax=ax, color='C0')
    df_contextquery[5].plot(ax=ax, color='C1')
    df_currentmotor[5].plot(ax=ax, color='C1')
    #axes[0].set_ylim(0,3.5)
    ax.set_ylabel('Current [A]')
    ax.set_xlabel('Error timestamps')
    ax.set_title(f'Raw Current')
    ax.axvline(x=end_time, color='C3', linestyle='-', label='Error Timestamp')
    ax.axvline(x=start_time, color='C2', linestyle='-')
    ax.grid(True)
    ax.legend(handles=None)
    plt.show()
    plt.clf

    ax = df_signal[4].plot(figsize=(15,10), color='C0')
    df_signal[5].plot(ax=ax, color='C1')
    ax.set_xlabel('Current [A]')
    ax.set_ylabel('Error Timestamps')
    ax.set_title(f'Zoom of Motors Current')
    ax.axvline(x=end_time, color='C3', linestyle='-', label=f'{end_time}')
    ax.axvline(x=start_time, color='C2', linestyle='-', label=f'{start_time}')
    ax.legend()
    ax.grid(True)
    plt.show()
    plt.clf
    
    ax = plt.figure(figsize=(15,10))
    plt.plot(freqs[4], real_fft_vals[4], color='C0', label=f'Motor 4')
    plt.scatter(freqs[4][peaks_index[4]],properties[4]['peak_heights'], 
                marker='x', s=100, color='C4', label='Peaks Motor 4')
    plt.plot(freqs[5], real_fft_vals[5], color='C1', label=f'Motor 5')  
    plt.scatter(freqs[5][peaks_index[5]],properties[5]['peak_heights'], 
                marker='x', s=100, color='C3', label='Peaks Motor 5')
    plt.xlabel('Frequency [Hz]')
    plt.ylabel('Power')
    plt.axvline(x=(1/dt)/2, color='C3', linestyle='-', label=f'Nyquist frequency = {(1/dt)/2}')
    plt.xlim(0,)
    plt.title(f'Real part of FFT')
    plt.legend()
    plt.grid(True)
    plt.show()
    plt.clf
    
    ax = plt.figure(figsize=(15,10))
    plt.plot(freqs[4], imag_fft_vals[4], color='C0', label=f'Motor 4')
    plt.plot(freqs[5], imag_fft_vals[5], color='C1', label=f'Motor 5')  
    plt.xlabel('Frequency [Hz]')
    plt.ylabel('Power')
    plt.xlim(0,)
    plt.title(f'Imaginary part of FFT')
    plt.legend()
    plt.grid(True)
    plt.show()
    plt.clf

With this example we can see a model analysis, with two dominant frequencies from the real FFT\
In the Signal we also can see the inner and outer oscillations of the current

## Multiple timestamps

In [ ]:
print('Amount of timestamps:', len(timestamps_error_code))

### Warning long running time
Consider the amount of timestamps increases by around 5 seconds the amount of time the next cell takes to process

In [ ]:
plt.close()

df_contextquery = {}
df_currentmotor = {}
peaks_index = {}
properties = {}
df_signal = {}
fft_vals = {}
real_fft_vals = {}
imag_fft_vals = {}
freqs = {}
power = {}

for timestamp in timestamps_error_code:
    
    context_alt, context_az, error_alt, error_az = alt_az_query(delta_start, delta_end, timestamp)

    df_context_alt = await efd_client.influx_client.query(context_alt)
    df_error_alt = await efd_client.influx_client.query(error_alt)

    for motor in range(4,6):
        
        fft_query, context_query, start_time, end_time = current_query(motor, delta_start, delta_end, timestamp)

        df_contextquery[motor] = await efd_client.influx_client.query(context_query)
        df_currentmotor[motor] = await efd_client.influx_client.query(fft_query)

        dt = dt_median
    
        (real_fft_vals[motor], imag_fft_vals[motor],
         freqs[motor], peaks_index[motor], df_signal[motor]) = fft_positive_values(motor, df_currentmotor[motor], dt)

            
    if verbose:
        
        print(f'Time: {timestamp}')
        
        ax = df_contextquery[4].plot(figsize=(15,10), color='C0')
        df_currentmotor[4].plot(ax=ax, color='C0')
        df_contextquery[5].plot(ax=ax, color='C1')
        df_currentmotor[5].plot(ax=ax, color='C1')
        ax.set_ylabel('Current [A]')
        ax.set_xlabel('Error timestamps')
        ax.set_title(f'Raw Current')
        ax.axvline(x=end_time, color='C3', linestyle='-', label='Error Timestamp')
        ax.axvline(x=start_time, color='C2', linestyle='-')
        ax.grid(True)
        ax.legend(handles=None)
        plt.show()
        plt.clf
        
        ax = df_signal[4].plot(figsize=(15,10), color='C0')
        df_signal[5].plot(ax=ax, color='C1')
        ax.set_xlabel('Current [A]')
        ax.set_ylabel('Error Timestamps')
        ax.set_title(f'Zoom of Motors Current')
        ax.axvline(x=end_time, color='C3', linestyle='-', label=f'{end_time}')
        ax.axvline(x=start_time, color='C2', linestyle='-', label=f'{start_time}')
        ax.legend()
        ax.grid(True)
        plt.show()
        plt.clf
        
        ax = plt.figure(figsize=(15,10))
        plt.plot(freqs[4], real_fft_vals[4], color='C0', label=f'Motor 4')
        plt.scatter(freqs[4][peaks_index[4]],properties[4]['peak_heights'], 
                    marker='x', s=100, color='C4', label='Peaks Motor 4')
        plt.plot(freqs[5], real_fft_vals[5], color='C1', label=f'Motor 5')  
        plt.scatter(freqs[5][peaks_index[5]],properties[5]['peak_heights'], 
                    marker='x', s=100, color='C3', label='Peaks Motor 5')
        plt.xlabel('Frequency [Hz]')
        plt.ylabel('Power')
        plt.axvline(x=(1/dt)/2, color='C3', linestyle='-', label=f'Nyquist frequency = {(1/dt)/2}')
        plt.xlim(0,)
        plt.title(f'Real part of FFT')
        plt.legend()
        plt.grid(True)
        plt.show()
        plt.clf
        
        ax = plt.figure(figsize=(15,10))
        plt.plot(freqs[4], imag_fft_vals[4], color='C0', label=f'Motor 4')
        plt.plot(freqs[5], imag_fft_vals[5], color='C1', label=f'Motor 5')  
        plt.xlabel('Frequency [Hz]')
        plt.ylabel('Power')
        plt.xlim(0,)
        plt.title(f'Imaginary part of FFT')
        plt.legend()
        plt.grid(True)
        plt.show()
        plt.clf

In [ ]:
df_currentmotor = {}
peaks_index = {}
properties = {}
real_fft_vals = {}
freqs = {}
power = {}

plt.clf()
fig = plt.plot(figsize=(10,3))

for timestamp in timestamps_error_code:
    for motor in range(4,6):
        
        fft_query, context_query, start_time, end_time = current_query(motor, delta_start, 
                                                                       delta_end, timestamp)

        df_currentmotor[motor] = await efd_client.influx_client.query(fft_query)
        dt = dt_median

        (real_fft_vals[motor], imag_fft_vals[motor],
         freqs[motor], peaks_index[motor], df_signal[motor]) = fft_positive_values(motor, df_currentmotor[motor], dt)

        if motor == 4:
            color = 'C0'
            alpha = 0.5
        if motor == 5:
            color = 'C1'
            alpha = 0.25
        
        plt.plot(freqs[motor], real_fft_vals[motor], alpha=0.2, color=color)
        #plt.scatter(freqs[motor][peaks_index[motor]],properties[motor]['peak_heights'], marker='x', alpha=0.2, s=100,)
        
plt.xlabel('Frequency [Hz]')
plt.ylabel('Power')
plt.xlim(0,) # Outside of these limits there are no useful frequencies
plt.title('Population plot of FFTs')
plt.grid(True)
plt.show()

The next part of the code, gathers the dominant frequencies for both Struts 4 and 5 and it attaches them to the same 1st Dominant and 2nd Dominant Frequency lists, without discriminating to which strut it corresponds to, but considering both Struts excert similar frequencies for each instance, this can be overlooked to gather a Table with same-sized lists

In [ ]:
delta = 1

df_currentmotor = {}
peaks_index = {}
properties = {}
fft_vals = {}
real_fft_vals = {}
freqs = {}
first_peak = {}
second_peak = {}

first_peak[4] = []
second_peak[4] = []
first_peak[5] = []
second_peak[5] = []
timestamp_list = []
altitude_list = []


for timestamp in timestamps_error_code:
    
    context_alt, context_az, error_alt, error_az = alt_az_query(delta, delta, timestamp)
    df_error_alt = await efd_client.influx_client.query(error_alt)
    
    altmedian = df_error_alt[f'Elevation'].median()
    
    for motor in range(4,6):
        
        fft_query, context_query, start_time, end_time = current_query(motor, delta_start, delta_end, timestamp)
        
        df_currentmotor[motor] = await efd_client.influx_client.query(fft_query)
        dt = dt_median
        
        (real_fft_vals[motor], imag_fft_vals[motor],
         freqs[motor], peaks_index[motor], df_signal[motor]) = fft_positive_values(motor, df_currentmotor[motor], dt)
        
    if len(peaks_index[4]) < 2 or len(peaks_index[5]) < 2:
        continue
    
    min_hz_distance = 0.0 # This should not be 0
    
    for motor in range(4,6):
        chosen_peaks = []
        chosen_peaks.append(int(peaks_index[motor][-1]))
        for peak in peaks_index[motor][:-1]:
            if freqs[motor][chosen_peaks] - freqs[motor][peak] > min_hz_distance: # This is always true, how do I make sure I'm doing this right?
                chosen_peaks.append(int(peak))
            if len(chosen_peaks) == 2:
                f1 = freqs[motor][chosen_peaks[0]]
                f2 = freqs[motor][chosen_peaks[1]]
                first_peak[motor].append(f1)
                second_peak[motor].append(f2)
                break
    
    timestamp_list.append(timestamp)
    altitude_list.append(altmedian)

print(f'All same length = {len(timestamp_list)}?', print(len(timestamp_list) == len(first_peak[4])  == len(first_peak[5])))

In [ ]:
df = pd.DataFrame({
        'Time': timestamp_list,
        'Elevation': altitude_list,
        '1st Frequency Motor 5': first_peak[4],
        '2nd Frequency Motor 5': second_peak[4],
        '1st Frequency Motor 6': first_peak[5],
        '2nd Frequency Motor 6': second_peak[5]
    })
df['Time'] = pd.to_datetime(df['Time'])
df.set_index('Time', inplace=True)
elevation_frequencies = df
matrix = elevation_frequencies.corr()

cmap = mpl.cm.Spectral
bounds = np.linspace(-1, 1, 9)
norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend='both')

plt.figure(figsize=(10,6))
sb.heatmap(matrix, annot=True, cmap=cmap, center=0.2, norm=norm)
plt.title('Correlation Matrix')
plt.xticks(rotation=45)
plt.show()
plt.clf

elevation_frequencies.plot(figsize=(10,6))
plt.title('Plot of Dominant Frequencies and Elevation')
plt.ylim(0,20)
plt.grid(True)
plt.show()

In [ ]:
df

What does this mean?\
We also should check whether the motors have the same vibrations on a same day (maybe use another matrix?)

- Create a document with the plots
- Dominique: Same or different frequencies on same day?
- Karla Pena Obs Specialist
- Wrap up
- Create a logger
- Since Jan 1 2024 or 2021
- (-- ammend) (-- fixup)